# 06 — Deduplicación y fusión: obras, manifestaciones y autores UNAM

**Primera ejecución: diagnóstico A–Q, sin eliminar filas ni asignar índices finales.**
Se trabaja únicamente con `../04_Limpieza/03_limpieza_bibliografica/autores_unam_limpios.csv`.
No se consulta internet. Las antiguas bases deduplicadas y decisiones por fuente no son entradas.

## Reglas de esta versión

`Fuente_origen` se elimina de la tabla de trabajo antes de indexar; no es frontera ni preferencia.
El `indice` histórico sirve solo para auditoría. Los diez campos de artículo generan **perfiles
auxiliares** que conservan todos sus registros originales. Un perfil no equivale a una manifestación confirmada.
Se comparan perfiles mediante bloqueo por título/DOI/abstract y vecinos de q-gramas. Las reglas
bibliográficas se aplican antes de los scores; un DOI vacío nunca sirve para unir dos DOI distintos.

Hay dos scores: `score_obra` y `score_manifestacion`. Sus pesos por estado se estiman dentro del archivo
con razones de verosimilitud suavizadas. Las familias se separan para ajuste/validación. Las etiquetas
iniciales son **semillas heurísticas**, no una verdad de terreno independiente: una aparente precisión
perfecta sobre esas semillas no valida la exactitud bibliográfica ni autoriza fusiones de casos nuevos.
Los umbrales propuestos y la distribución de features se exportan para revisión. No se adopta un 0.90 arbitrario.
Cinco vecinos es un presupuesto de recuperación, no un criterio de identidad; su cobertura tampoco es recall medido.

## Archivos generados en `04_Limpieza/04_deduplicacion/`

- `diagnostico_deduplicacion.csv`: A–Q, faltantes, conflictos, estimaciones y huella de entrada.
- `candidatos_deduplicacion.csv`: features, dos scores, hard rules y evidencia de cada par.
- `plan_manifestaciones.csv`: perfiles y asignaciones de obra/manifestación propuestas; contiene los valores originales completos.
- `casos_revision_deduplicacion.csv`: revisión agrupada de identidad y conflictos por campo.
- `calibracion_deduplicacion.csv`: pesos empíricos, estados, distribuciones, límites propuestos y sus restricciones.
- `potenciales_propagaciones.csv`: oportunidades internas; no son cambios aplicados.

Las listas `row_ids_*` contienen **posiciones de registros de datos empezando en 1**, sin encabezado:
no son los índices históricos ni los números físicos de línea de un CSV con saltos dentro de celdas.
No editar con inferencia numérica de Excel: ISBN/ISSN/índices deben permanecer como texto.

## Cómo revisar y después ejecutar la fusión

Primero ejecutar `modo = "DIAGNOSTICO"`. No hay que resolver automáticamente los antiguos 182 casos.
Los conteos de manifestaciones/filas son propuestas, no objetivos que deba forzarse a reproducir.

**1. Resolver identidad en el plan.** Editar únicamente `Obra_manual`, `Manifestacion_manual`
y `Comentario_manual`. Los identificadores pueden ser etiquetas locales (p. ej. `O_REV_01`, `M_REV_01`)
o etiquetas propuestas existentes. Igual obra y distinta manifestación se representan con la MISMA etiqueta
de obra y DIFERENTES etiquetas de manifestación. Distintos autores no se colapsan en una fila.
Dejar los campos manuales vacíos mantiene la propuesta. Una manifestación no puede tener dos obras.

Reejecutar DIAGNOSTICO tras cambiar identidades, **antes de aprobar los casos**: el nuevo plan puede
hacer aparecer o desaparecer conflictos y oportunidades. El programa no traslada aprobaciones a
contextos diferentes. Resguardar las decisiones antiguas si se requiere reorganizar un plan ya aprobado.

**2. Resolver campos y transferencias en los casos.** Editar solo `Decision_manual`,
`row_ids_evidencia` y `Comentario_manual`. Cada fila muestra `Acciones_permitidas`.
Las referencias se escriben como `12 | 35`, usando exclusivamente registros del ámbito del caso:

- `VALIDAR_PLAN`: aprobar la relación obra/manifestación definida en el plan, no fusionar por sí solo.
- `SELECCIONAR`: elegir un valor existente; referenciar filas con ese mismo valor.
- `COMBINAR`: ISBN, ISSN o afiliaciones confirmadas como compatibles; cada componente debe existir en los donantes.
- `CORREGIR_DOI_INTERNO`: solo para un DOI claramente erróneo demostrado dentro del archivo; es una aprobación explícita, nunca automática.
- `PROPAGAR` / `NO_PROPAGAR`: decidir una incorporación de autor o precisión de afiliación de ese MISMO autor y obra.
- `ARMONIZAR_CONTENIDO` / `NO_ARMONIZAR_VERSIONES`: resolver la precaución frente a versiones/preprints.
- `MANTENER_POR_VERSION`: conservar año/abstract por versión cuando está justificado; no sirve para escoger un área por mayoría.
- `MANTENER_POR_MANIFESTACION`: no propagar afiliaciones específicas contradictorias entre versiones.

Todas las decisiones requieren evidencia interna y comentario. No introducir valores nuevos en los campos
protegidos del plan o de los casos. Un autor solo se agrega entre manifestaciones tras aprobación
específica; que exista en el archivo no demuestra que falte en la versión receptora.

**3. Solo después de aprobar el diagnóstico y cerrar TODOS los casos**, configurar:

```python
modo = "FUSION_FINAL"
aprobar_diagnostico = True
sha256_diagnostico_aprobado = "<huella del archivo que se aprobó>"
```

La fusión genera `autores_unam_deduplicados.csv` (14 columnas), `auditoria_fusion.csv` (linaje de cada
relación final) y `auditoria_campos_fusion.csv` (donantes por campo/componente). Con cualquier conflicto
sin resolver se detiene: no entrega una base parcial como definitiva. En DIAGNOSTICO no cambia el
archivo deduplicado histórico que pudiera existir en la carpeta.

Los nuevos índices siguen la primera aparición de cada manifestación en la entrada; todos sus autores
comparten índice. Hay una fila por autor exacto normalizado y manifestación. No se cambian nombres,
no se agregan instituciones ajenas a las relaciones de entrada y no se usan afiliaciones de otro autor.
Los ISBN/ISSN/DOI/URL solo se combinan dentro de su manifestación; contenido y año pueden armonizarse
en la obra con sus controles. No se concatenan abstracts.

`actualizar_archivos = False` protege resultados diferentes. Antes de sustituirlos, resguardarlos con
GitHub Desktop y autorizar `True`. Los archivos de entrada nunca se sobrescriben. Se releen todos los CSV.

### Material metodológico proporcionado

Métodos de indexado (bloqueo/q-gramas); métodos de comparación (Levenshtein y comparación por términos);
métodos de clasificación (pesos y dos umbrales, limitaciones de sumar atributos); evaluación de
clasificación (precisión/recall y falsos positivos); fusión de datos (incertidumbre frente a contradicción,
selección por celda y linaje); introducción a FEBRL (organización del flujo). Se conserva esa separación
de tareas sin adoptar sus ejemplos numéricos como parámetros universales.


In [2]:
# ============================================================
# 06 - DEDUPLICACIÓN Y FUSIÓN: OBRA -> MANIFESTACIÓN -> AUTOR
# Solo datos internos. Fuente_origen e indice NO son features.
# ============================================================
import csv
import hashlib
import io
import json
import math
import re
import unicodedata
from collections import Counter, defaultdict
from functools import lru_cache
from itertools import combinations
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit, unquote

import numpy as np
import pandas as pd

# ============================================================
# 1. CONFIGURACIÓN - MISMAS CARPETAS DEL PROYECTO
# ============================================================
RAIZ_MANUAL = None                 # None: localizar Tesis_Multimodelo desde notebooks/
modo = "DIAGNOSTICO"               # DIAGNOSTICO o FUSION_FINAL
actualizar_archivos = True       # Resguardar en Git antes de autorizar reemplazos.
aprobar_diagnostico = False        # Solo activar DESPUÉS de revisar diagnóstico A-Q.
sha256_diagnostico_aprobado = ""    # Copiar huella de la entrada que se aprobó.
vecinos_titulo = 5                 # Recuperación de candidatos, NO umbral de fusión.

CANON = ['indice','Titulo','Año','Autor_norm','Afiliacion1','Afiliacion2',
         'ISBN','ISSN','Doi','URL','Area','SubArea','Keywords','Abstract']
BIB = ['Titulo','Año','ISBN','ISSN','Doi','URL','Area','SubArea','Keywords','Abstract']
AREAS = {'CC','IA','ISBD','RS','SIAV','TC'}
GENERICA = 'Universidad Nacional Autónoma de México'
SCI = re.compile(r'^[+-]?\d+(?:\.\d+)?[eE][+-]?\d+$')
DOI = re.compile(r'^10\.\d{4,9}/\S+$')
VERSION = re.compile(r'(?:10\.48550/arxiv\.|10\.1101/|10\.21203/|arxiv\.org/|biorxiv\.org/|medrxiv\.org/|\bpreprint\b|\bextended version\b|\bjournal extension\b|\bcorrigendum\b|\berratum\b)',re.I)


def js(x):
    return json.dumps(x, ensure_ascii=False, separators=(',',':'), sort_keys=True)

def huella_objeto(x):
    return hashlib.sha256(js(x).encode('utf-8')).hexdigest()

def texto(x):
    return re.sub(r'\s+',' ',unicodedata.normalize('NFC',str(x))).strip()

def norm_titulo(x):
    # Solo comparación: no cambia los títulos guardados. Conserva símbolos matemáticos.
    s=texto(x).casefold()
    return ' '.join(''.join(c if (c.isalnum() or unicodedata.category(c).startswith('S')) else ' ' for c in s).split())

def norm_abstract(x):
    return texto(x).casefold()

def norm_doi(x):
    s=texto(x).casefold()
    return re.sub(r'^(?:https?://(?:dx\.)?doi\.org/|doi\s*:\s*)','',s)

def norm_url(x):
    s=texto(x)
    if not s:return ''
    u=urlsplit(s)
    host=(u.hostname or '').lower()
    # No se pasa a minúsculas el path/query: pueden ser sensibles al caso.
    if host in {'doi.org','dx.doi.org'}:
        return 'https://doi.org/'+unquote(u.path.lstrip('/')).casefold().rstrip('/')
    net=host
    if u.port and not ((u.scheme.lower()=='https' and u.port==443) or (u.scheme.lower()=='http' and u.port==80)):
        net+=':'+str(u.port)
    return urlunsplit((u.scheme.lower(),net,u.path.rstrip('/'),u.query,''))

def doi_url(x):
    u=urlsplit(texto(x)) if texto(x) else None
    return unquote(u.path.lstrip('/')).casefold().rstrip('/') if u and (u.hostname or '').lower() in {'doi.org','dx.doi.org'} else ''

def tokens_celda(x):
    return [v.strip() for v in str(x).split(';') if v.strip()]

def norm_id(x):
    return re.sub(r'[-\s]','',x).upper()

def isbn_valido(x):
    s=norm_id(x)
    if re.fullmatch(r'\d{9}[\dX]',s):
        return sum((10-i)*(10 if c=='X' else int(c)) for i,c in enumerate(s))%11==0
    if re.fullmatch(r'97[89]\d{10}',s):
        return sum((1 if i%2==0 else 3)*int(c) for i,c in enumerate(s))%10==0
    return False

def issn_valido(x):
    s=norm_id(x)
    return bool(re.fullmatch(r'\d{7}[\dX]',s)) and sum((8-i)*(10 if c=='X' else int(c)) for i,c in enumerate(s))%11==0

def url_valida(x):
    if not x:return True
    try:
        u=urlsplit(x)
        return u.scheme in {'http','https'} and bool(u.hostname) and not re.search(r'\s',x) and u.username is None
    except ValueError:return False

def leer_csv(ruta):
    p=Path(ruta)
    if p.read_bytes()[:50].startswith(b'version https://git-lfs'):
        raise ValueError(f'{p.name} es un puntero Git LFS, no un CSV de datos.')
    return pd.read_csv(p,dtype=str,keep_default_na=False,encoding='utf-8-sig')

def validar_base(df, entrada=True):
    esquema=(['Fuente_origen']+CANON) if 'Fuente_origen' in df else CANON
    if list(df.columns)!=esquema:raise ValueError(f'Esquema inesperado: {list(df.columns)}')
    if not entrada and 'Fuente_origen' in df:raise ValueError('Fuente_origen no puede estar en la salida.')
    if df.empty:raise ValueError('La base está vacía.')
    for c in ['indice','Titulo','Autor_norm','Afiliacion1']:
        if df[c].str.strip().eq('').any():raise ValueError(f'{c} contiene vacíos.')
    if not set(df['Año']) <= {'','2024','2025'}:raise ValueError('Año fuera del checkpoint limpio.')
    if not set(df['Area']) <= AREAS:raise ValueError('Area inválida.')
    if df['SubArea'].ne('').any():raise ValueError('SubArea debe estar vacía.')
    for campo,fn in [('ISBN',isbn_valido),('ISSN',issn_valido)]:
        for i,v in df[campo].items():
            for t in tokens_celda(v):
                if SCI.fullmatch(t):raise ValueError(f'{campo} científico en registro {i+1}; no se reconstruye.')
                if not fn(t):raise ValueError(f'{campo} inválido en registro {i+1}: {t!r}')
    for i,v in df['Doi'].items():
        if v and (not DOI.fullmatch(v) or v!=norm_doi(v)):
            raise ValueError(f'DOI no limpio en registro {i+1}: {v}')
    for i,v in df['URL'].items():
        if not url_valida(v):raise ValueError(f'URL inválida en registro {i+1}')
    for i,v in df['Keywords'].items():
        t=[texto(s).casefold() for s in tokens_celda(v)]
        if len(t)!=len(set(t)):raise ValueError(f'Keywords repetidas internamente: registro {i+1}')
    if ((df.Afiliacion1==df.Afiliacion2)&df.Afiliacion2.ne('')).any():
        raise ValueError('Afiliaciones idénticas en ambas columnas.')


def cargar(ruta):
    raw=leer_csv(ruta);validar_base(raw)
    # Se elimina de la tabla de trabajo ANTES de indexar. Nunca es llave ni preferencia.
    df=raw.loc[:,CANON].copy()
    sha=hashlib.sha256(Path(ruta).read_bytes()).hexdigest()
    return df,sha


def unicos(values, key=lambda x:x):
    out=[];seen=set()
    for v in values:
        if not v:continue
        k=key(v)
        if k not in seen:seen.add(k);out.append(v)
    return out

@lru_cache(maxsize=10000)
def gramas(s,q=3):
    return frozenset(s[i:i+q] for i in range(max(0,len(s)-q+1)))

def jaccard(a,b):
    return len(a&b)/len(a|b) if a and b else 0.

@lru_cache(maxsize=50000)
def lev(a,b):
    # Distancia de Levenshtein exacta mediante vectores de bits. Los enteros Python
    # admiten cualquier longitud; devuelve el mismo 1-d/max_len que la matriz DP.
    if a==b:return 1.
    if not a or not b:return 0.
    if len(a)<len(b):a,b=b,a
    n=len(b);mask=(1<<n)-1;last=1<<(n-1);peq={}
    for i,ch in enumerate(b):peq[ch]=peq.get(ch,0)|(1<<i)
    positive=mask;negative=0;distance=n
    for ch in a:
        equal=peq.get(ch,0);xv=equal|negative
        xh=(((equal&positive)+positive)^positive)|equal
        ph=negative|~(xh|positive);mh=positive&xh
        if ph&last:distance+=1
        elif mh&last:distance-=1
        ph=(ph<<1)|1;mh<<=1
        positive=(mh|~(xv|ph))&mask;negative=(ph&xv)&mask
    return 1.-distance/max(len(a),len(b))


def abstract_mayor(values):
    vals=unicos(values, norm_abstract)
    if not vals:return '',True,'VACIO'
    # Un texto puede seleccionarse si TODOS los demás son iguales o están contenidos
    # literalmente (NFC/espacios/casefold) en él. No se usa solo longitud/similitud.
    ordered=sorted(vals,key=lambda v:(-len(norm_abstract(v)),vals.index(v)))
    for candidate in ordered:
        cn=norm_abstract(candidate)
        ok=True
        for v in vals:
            vn=norm_abstract(v)
            if vn==cn:continue
            core=vn.rstrip(' .…')
            if len(core.split())<12 or core not in cn:ok=False;break
        if ok:return candidate,True,'IGUAL_O_CONTENCION_LITERAL'
    return '',False,'TEXTOS_DIFERENTES'


def construir_representaciones(df):
    # Auxiliar sin destrucción: cada perfil de los 10 campos de artículo guarda TODOS
    # sus row_ids. Ni indice ni Fuente_origen intervienen en estas firmas.
    groups={}
    for pos,r in enumerate(df.to_dict('records'),1):
        sig=tuple(r[c] for c in BIB)
        groups.setdefault(sig,[]).append(pos)
    reps=[]
    for sig,ids in groups.items():
        r=dict(zip(BIB,sig));r['rid']='B_'+huella_objeto(list(sig))[:16].upper()
        r['row_ids']=ids;r['first']=min(ids)
        r['authors']=frozenset(df.iloc[[i-1 for i in ids]].Autor_norm)
        r['tn']=norm_titulo(r['Titulo']);r['dn']=norm_doi(r['Doi'])
        r['un']=norm_url(r['URL']);r['url_doi']=doi_url(r['URL'])
        r['isbn']=frozenset(norm_id(t) for t in tokens_celda(r['ISBN']))
        r['issn']=frozenset(norm_id(t) for t in tokens_celda(r['ISSN']))
        r['kn']=frozenset(texto(t).casefold() for t in tokens_celda(r['Keywords']))
        r['an']=norm_abstract(r['Abstract']);r['at']=frozenset(re.findall(r'\w+',r['an']))
        r['version']=bool(VERSION.search(r['Doi']+' '+r['URL']+' '+r['Titulo']))
        r['doi_url_conflict']=bool(r['dn'] and r['url_doi'] and r['dn']!=r['url_doi'])
        reps.append(r)
    return sorted(reps,key=lambda r:r['first'])


def generar_candidatos(reps, vecinos=5):
    # Blocking exacto DOI, título, abstract. Q-gramas para los k vecinos por título.
    pairs=defaultdict(set);by_t=defaultdict(list);by_d=defaultdict(list);by_a=defaultdict(list)
    for i,r in enumerate(reps):
        by_t[r['tn']].append(i)
        if r['dn']:by_d[r['dn']].append(i)
        if len(r['an'].split())>=12:by_a[r['an']].append(i)
    for reason,blocks in [('TITULO_EXACTO',by_t),('DOI_IGUAL',by_d),('ABSTRACT_IGUAL',by_a)]:
        for inds in blocks.values():
            for p in combinations(inds,2):pairs[p].add(reason)
    titles=sorted(by_t);post=defaultdict(set);tt={t:gramas(t) for t in titles}
    for t in titles:
        for q in tt[t]:post[q].add(t)
    nearest=[]
    for t in titles:
        hits=Counter(u for q in tt[t] for u in post[q] if u!=t)
        ranked=sorted(hits,key=lambda u:(-hits[u]/len(tt[t]|tt[u]),u))[:vecinos]
        for rank,u in enumerate(ranked,1):
            sim=jaccard(tt[t],tt[u]);nearest.append((t,u,rank,sim))
            for i in by_t[t]:
                for j in by_t[u]:pairs[tuple(sorted((i,j)))].add('TITULO_QGRAM_VECINOS')
    # Ejemplos internos negativos de calibración: pares de títulos distintos de un
    # mismo autor. Se conserva solo un perfil por título para no sobrerrepresentar.
    author_titles=defaultdict(dict)
    for i,r in enumerate(reps):
        for a in r['authors']:author_titles[a].setdefault(r['tn'],i)
    for mapping in author_titles.values():
        for i,j in combinations(mapping.values(),2):
            if reps[i]['dn'] and reps[j]['dn'] and reps[i]['dn']!=reps[j]['dn']:
                pairs[tuple(sorted((i,j)))].add('CONTROL_CALIBRACION')
    return pairs,nearest


def estado(a,b):
    if not a and not b:return 'DESCONOCIDO'
    if not a or not b:return 'UNO_VACIO'
    return 'IGUAL' if a==b else 'DISTINTO'

def estado_set(a,b):
    if not a and not b:return 'DESCONOCIDO'
    if not a or not b:return 'UNO_VACIO'
    if a==b:return 'IGUAL'
    return 'INTERSECCION' if a&b else 'DISJUNTOS'


def features_par(a,b):
    exact=a['tn']==b['tn'];ts=lev(a['tn'],b['tn'])
    abeq=bool(a['an'] and a['an']==b['an'])
    abshort=bool(a['an'] and b['an'] and abstract_mayor([a['Abstract'],b['Abstract']])[1])
    abgram=jaccard(gramas(a['an'],5),gramas(b['an'],5)) if a['an'] and b['an'] else 0.
    namesa,namesb=a['authors'],b['authors']
    num_a=re.findall(r'\d+(?:[.]\d+)?',a['tn']);num_b=re.findall(r'\d+(?:[.]\d+)?',b['tn'])
    doi_st=estado(a['dn'],b['dn']);ys=estado(a['Año'],b['Año'])
    same_url=bool(a['un'] and a['un']==b['un'])
    # No se extrae un DOI para rellenar: solo se compara el identificador explícito
    # dentro de una URL doi.org con el Doi del otro registro.
    url_support=bool((a['url_doi'] and a['url_doi']==b['dn']) or (b['url_doi'] and b['url_doi']==a['dn']))
    return {
        'doi_equal':int(doi_st=='IGUAL'),'doi_conflict':int(doi_st=='DISTINTO'),
        'doi_state':doi_st,'isbn_state':estado_set(a['isbn'],b['isbn']),
        'isbn_overlap':jaccard(a['isbn'],b['isbn']),
        'isbn_conflict':int(bool(a['isbn'] and b['isbn'] and not a['isbn']&b['isbn'])),
        'issn_state':estado_set(a['issn'],b['issn']),
        'issn_overlap':jaccard(a['issn'],b['issn']),
        'issn_conflict':int(bool(a['issn'] and b['issn'] and not a['issn']&b['issn'])),
        'url_state':estado(a['un'],b['un']),'url_equal':int(same_url),
        'url_doi_support':int(url_support),
        'title_exact':int(exact),'title_similarity':ts,
        'title_qgram':jaccard(gramas(a['tn']),gramas(b['tn'])),
        'title_tokens':jaccard(frozenset(a['tn'].split()),frozenset(b['tn'].split())),
        'title_numbers_conflict':int(num_a!=num_b),
        'year_equal':int(ys=='IGUAL'),'year_conflict':int(ys=='DISTINTO'),'year_state':ys,
        'author_overlap':jaccard(namesa,namesb),
        'author_common':len(namesa&namesb),'authors_disjoint':int(not namesa&namesb),
        'abstract_equal':int(abeq),'abstract_containment':int(abshort),
        'abstract_similarity':abgram,
        'abstract_state':'DESCONOCIDO' if not a['an'] and not b['an'] else 'UNO_VACIO' if not a['an'] or not b['an'] else 'IGUAL' if abeq else 'CONTENIDO' if abshort else 'DISTINTO',
        'keywords_overlap':jaccard(a['kn'],b['kn']),
        'keywords_state':estado_set(a['kn'],b['kn']),
        'area_conflict':int(a['Area']!=b['Area']),
        'version_risk':int(a['version']!=b['version'] or (a['dn']!=b['dn'] and (a['version'] or b['version']))),
        'doi_url_internal_conflict':int(a['doi_url_conflict'] or b['doi_url_conflict']),
        'evidence_observed':sum(bool(a[k] and b[k]) for k in ['dn','isbn','issn','un','tn','authors','an','kn'])
    }


def comparar(reps,pairs):
    rows=[]
    for (i,j),reasons in sorted(pairs.items()):
        a,b=reps[i],reps[j];f=features_par(a,b)
        r={'Par_ID':'P_'+huella_objeto(sorted([a['rid'],b['rid']]))[:16].upper(),
           'i':i,'j':j,'Representacion_A':a['rid'],'Representacion_B':b['rid'],
           'Bloqueo':'; '.join(sorted(reasons)),
           'Candidato_operativo':int(bool(reasons-{'CONTROL_CALIBRACION'})),**f}
        # Semillas internas/provisionales, NO verdad de terreno ni validación externa.
        clear_pos=bool(f['doi_equal'] and f['title_exact'] and f['author_common'])
        clear_neg_work=bool(f['doi_conflict'] and not f['title_exact'] and
            not (set(a['tn'].split())&set(b['tn'].split())-{'a','an','the','of','in','for','and','with','using','on','to'})
            and not f['abstract_containment'])
        r['Semilla_manifestacion']='POSITIVA' if clear_pos else 'NEGATIVA' if f['doi_conflict'] else ''
        r['Semilla_obra']='POSITIVA' if clear_pos else 'NEGATIVA' if clear_neg_work else ''
        r['Familia_A']=a['tn'];r['Familia_B']=b['tn']
        rows.append(r)
    cols=['Par_ID','i','j','Representacion_A','Representacion_B','Bloqueo','Candidato_operativo']+list(features_par(reps[0],reps[0]))+['Semilla_manifestacion','Semilla_obra','Familia_A','Familia_B']
    return pd.DataFrame(rows,columns=cols)

# ============================================================
# CALIBRACIÓN EXPLORATORIA: pesos empíricos de estados por campo
# ============================================================

def particion_calibracion(pares,reps):
    by_d=defaultdict(list)
    for r in reps:
        if r['dn']:by_d[r['dn']].append(r['tn'])
    fam={r['rid']:min(by_d[r['dn']]) if r['dn'] else r['tn'] for r in reps}
    fold={rid:int(huella_objeto(v)[:8],16)%5 for rid,v in fam.items()}
    out=[]
    for r in pares.to_dict('records'):
        a=fold[r['Representacion_A']]==0;b=fold[r['Representacion_B']]==0
        out.append('VALIDACION_INTERNA' if a and b else 'AJUSTE' if not a and not b else 'ENTRE_PARTICIONES')
    return out


def calibrar(pares,reps):
    pares=pares.copy();pares['Particion']=particion_calibracion(pares,reps)
    # Cortes descriptivos sobre AJUSTE; no son umbrales de match.
    cortes={}
    numeric=['title_similarity','author_overlap','abstract_similarity','keywords_overlap']
    for c in numeric:
        v=pares.loc[pares.Particion.eq('AJUSTE') & pares[c].gt(0) & pares[c].lt(1),c].to_numpy(float)
        cortes[c]=sorted(set(float(x) for x in np.quantile(v,[.25,.5,.75]))) if len(v) else []
    def categoria(r,c):
        if c=='title':
            return 'EXACTO' if r['title_exact'] else 'Q'+str(np.searchsorted(cortes['title_similarity'],r['title_similarity'],side='right'))
        if c=='author':
            return 'SIN_INTERSECCION' if r['author_common']==0 else 'IGUALES' if r['author_overlap']==1 else 'INTERSECCION_Q'+str(np.searchsorted(cortes['author_overlap'],r['author_overlap'],side='right'))
        if c=='abstract':
            st=r['abstract_state']
            return st if st!='DISTINTO' else 'DIFERENTE_Q'+str(np.searchsorted(cortes['abstract_similarity'],r['abstract_similarity'],side='right'))
        if c=='keywords':
            st=r['keywords_state']
            return st if st!='INTERSECCION' else 'INTERSECCION_Q'+str(np.searchsorted(cortes['keywords_overlap'],r['keywords_overlap'],side='right'))
        return r[c+'_state']
    fields=['title','author','abstract','keywords','year','doi','isbn','issn','url']
    for f in fields:pares['estado_'+f]=[categoria(r,f) for r in pares.to_dict('records')]
    records=[];models={};thresholds={}
    for level in ['obra','manifestacion']:
        fs=['title','author','abstract','keywords','year']
        if level=='manifestacion':fs+=['doi','isbn','issn','url']
        seed='Semilla_'+level
        subset=pares.loc[pares.Particion.eq('AJUSTE')&pares[seed].ne('')].copy()
        family=[tuple(sorted((a,b))) for a,b in zip(subset.Familia_A,subset.Familia_B)]
        ct=Counter(family);subset['_peso_familia']=[1/ct[k] for k in family]
        model={}
        for f in fs:
            col='estado_'+f;states=sorted(set(pares[col]));K=len(states)
            totals={cl:float(subset.loc[subset[seed].eq(cl),'_peso_familia'].sum()) for cl in ['POSITIVA','NEGATIVA']}
            for state in states:
                cnt={cl:float(subset.loc[subset[seed].eq(cl)&subset[col].eq(state),'_peso_familia'].sum()) for cl in totals}
                p=(cnt['POSITIVA']+.5)/(totals['POSITIVA']+.5*K)
                q=(cnt['NEGATIVA']+.5)/(totals['NEGATIVA']+.5*K)
                w=math.log2(p/q) if state not in {'DESCONOCIDO','UNO_VACIO'} else 0.
                model[f,state]=w
                records.append({'Nivel':level.upper(),'Tipo':'PESO_ESTADO','Feature':f,'Estado':state,
                    'Peso_log2':w,'N_positivo_ponderado':cnt['POSITIVA'],'N_negativo_ponderado':cnt['NEGATIVA'],
                    'Valor':'','Detalle':'log2((n+0.5)/(N+0.5*K)); una unidad de peso por familia de títulos. Ausencia aporta 0. No es probabilidad.'})
        models[level]=model
        sc='score_'+level
        pares[sc]=[sum(model[f,r['estado_'+f]] for f in fs) for r in pares.to_dict('records')]
        val=pares.loc[pares.Particion.eq('VALIDACION_INTERNA')&pares[seed].ne('')].copy()
        pos=val.loc[val[seed].eq('POSITIVA'),sc];neg=val.loc[val[seed].eq('NEGATIVA'),sc]
        enabled=bool(len(pos) and len(neg))
        if enabled:
            high=float(min(pos)) if float(min(pos))>float(max(neg)) else float(np.nextafter(float(max(neg)),math.inf))
            low=min(float(min(pos)),float(max(neg)))
            thresholds[level]=(low,high)
            tp=int((pos>=high).sum());fp=int((neg>=high).sum())
            info={'positivos_validacion':len(pos),'negativos_validacion':len(neg),'familias_positivas_validacion':int(val.loc[val[seed].eq('POSITIVA'),'Familia_A'].nunique()),'familias_negativas_validacion':int(val.loc[val[seed].eq('NEGATIVA'),'Familia_A'].nunique()),'positivos_superan':tp,'negativos_superan':fp,
                  'precision_semillas':tp/(tp+fp) if tp+fp else None,'recall_semillas':tp/len(pos),
                  'limitacion':'Semillas definidas con el propio archivo; NO ground truth independiente. DOI distintos no son negativos de obra por sí solos.'}
        else:
            thresholds[level]=(None,None);info={'estado':'SIN_AMBAS_CLASES_EN_VALIDACION; no hay umbral utilizable'}
        records.append({'Nivel':level.upper(),'Tipo':'UMBRAL_PROPUESTO','Feature':'score_'+level,'Estado':'SUPERIOR',
             'Valor':thresholds[level][1],'Detalle':js(info)})
        records.append({'Nivel':level.upper(),'Tipo':'UMBRAL_PROPUESTO','Feature':'score_'+level,'Estado':'INFERIOR',
             'Valor':thresholds[level][0],'Detalle':'Zona intermedia = revisión. Los umbrales no vencen hard rules ni bastan para fusionar sin DOI.'})
        for scope in ['AJUSTE','VALIDACION_INTERNA']:
            for cl in ['POSITIVA','NEGATIVA','']:
                vals=pares.loc[pares.Particion.eq(scope)&pares[seed].eq(cl),sc]
                if len(vals):
                    records.append({'Nivel':level.upper(),'Tipo':'DISTRIBUCION_SCORE','Feature':sc,
                        'Estado':scope+'_'+(cl or 'SIN_ETIQUETA'),'Valor':len(vals),
                        'Detalle':js({str(k):float(v) for k,v in vals.quantile([0,.1,.25,.5,.75,.9,1]).items()})})
    for c,v in cortes.items():records.append({'Nivel':'AMBOS','Tipo':'CORTES_FEATURE','Feature':c,'Valor':js(v),
        'Detalle':'Cuartiles de valores no extremos observados en partición AJUSTE; solo discretización.'})
    for c in ['doi_equal','doi_conflict','isbn_overlap','isbn_conflict','issn_overlap','issn_conflict','url_equal',
              'title_exact','title_similarity','year_equal','year_conflict','author_overlap','abstract_similarity','keywords_overlap']:
        vals=pares[c]
        records.append({'Nivel':'AMBOS','Tipo':'DISTRIBUCION_FEATURE','Feature':c,'Valor':len(vals),
            'Detalle':js({str(k):float(v) for k,v in vals.quantile([0,.25,.5,.75,1]).items()})})
    return pares,pd.DataFrame(records).fillna(''),thresholds


# ============================================================
# HARD RULES Y CLASIFICACIÓN. Identidad != resolución de campos.
# ============================================================

def clasificar_pares(pares,reps,thresholds):
    out=[]
    low,high=thresholds['obra']
    for r in pares.to_dict('records'):
        a,b=reps[r['i']],reps[r['j']]
        reasons=[];hard_m=False
        if r['doi_equal']:
            obra='MISMA_OBRA' if r['title_exact'] else 'REVISION'
            man='MISMA_MANIFESTACION' if r['title_exact'] else 'REVISION'
            reason='DOI_IGUAL_TITULO_COMPATIBLE' if r['title_exact'] else 'DOI_IGUAL_TITULO_NO_EQUIVALENTE'
            hard_m=bool(r['title_exact'])
        elif r['title_exact']:
            strong_text=r['abstract_containment'] and r['author_common']>0
            obra='MISMA_OBRA' if strong_text else 'REVISION'
            if r['doi_conflict']:
                man='DISTINTA_MANIFESTACION';reason='DOI_DISTINTOS'
            else:
                # No basta que no haya conflicto. Exigir ancla de manifestación.
                anchor=bool(r['url_equal'] or r['url_doi_support'] or (r['isbn_overlap'] and r['abstract_containment'] and r['author_common']))
                idok=not r['isbn_conflict'] and not r['issn_conflict'] and not r['version_risk']
                hard_m=anchor and idok and not r['year_conflict']
                man='MISMA_MANIFESTACION' if hard_m else 'REVISION'
                reason='TITULO_ANCLA_INTERNA_SIN_CONTRADICCION' if hard_m else 'TITULO_IGUAL_MANIFESTACION_INCIERTA'
                if hard_m:obra='MISMA_OBRA'
        else:
            if r['abstract_equal'] and r['author_common']:
                obra='REVISION';reason='ABSTRACT_IGUAL_TITULO_DISTINTO'
            elif low is not None and r['score_obra']<low:
                obra='OBRA_DISTINTA';reason='BAJO_UMBRAL_EXPLORATORIO_SIN_IDENTIDAD_DURA'
            else:
                obra='REVISION';reason='TITULO_APROXIMADO_O_EVIDENCIA_INSUFICIENTE'
            man='DISTINTA_MANIFESTACION' if r['doi_conflict'] or obra=='OBRA_DISTINTA' else 'REVISION'
        if not r['Candidato_operativo'] and not r['doi_equal'] and not r['title_exact'] and not r['abstract_equal']:
            obra='CONTROL_NO_OPERATIVO';man='CONTROL_NO_OPERATIVO';reason='SOLO_CALIBRACION';hard_m=False
        if r['doi_conflict'] and r['title_exact'] and lev(a['dn'],b['dn'])>=1.-2/max(len(a['dn']),len(b['dn'])):
            reasons.append('DOI_DISTINTOS_POSIBLE_ERROR_LOCAL_NO_CORREGIDO')
        if r['doi_url_internal_conflict']:reasons.append('DOI_URL_CONTRADICTORIOS')
        if r['version_risk'] and obra!='OBRA_DISTINTA' and obra!='CONTROL_NO_OPERATIVO':
            reasons.append('POSIBLE_CAMBIO_DE_VERSION')
        if r['year_conflict'] and obra in {'MISMA_OBRA','REVISION'}:reasons.append('ANIO_DISTINTO')
        if r['area_conflict'] and obra in {'MISMA_OBRA','REVISION'}:reasons.append('AREA_DISTINTA')
        # DOI igual permite identificar aunque ISBN/ISSN sean distintos: conflicto a
        # examinar antes de fusionar, sin fragmentar artificialmente el DOI.
        if r['isbn_conflict'] and obra!='OBRA_DISTINTA':reasons.append('ISBN_DISJUNTOS')
        if r['issn_conflict'] and obra!='OBRA_DISTINTA':reasons.append('ISSN_DISJUNTOS')
        out.append({**r,'Decision_obra':obra,'Decision_manifestacion':man,
                    'Regla_match':reason,'Alertas':'; '.join(reasons),'Arista_manifestacion_segura':int(hard_m)})
    return pd.DataFrame(out,columns=list(dict.fromkeys(list(pares.columns)+['Decision_obra','Decision_manifestacion','Regla_match','Alertas','Arista_manifestacion_segura'])))


class Conjuntos:
    def __init__(self,n):self.p=list(range(n))
    def find(self,x):
        while self.p[x]!=x:self.p[x]=self.p[self.p[x]];x=self.p[x]
        return x
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a!=b:self.p[max(a,b)]=min(a,b)
    def grupos(self):
        groups=defaultdict(list)
        for i in range(len(self.p)):groups[self.find(i)].append(i)
        return list(groups.values())


def id_grupo(prefix,inds,reps):
    return prefix+'_'+huella_objeto(sorted(reps[i]['rid'] for i in inds))[:16].upper()


def agrupar_propuestas(reps,pares):
    # Grafo de candidatos -> consistencia de TODO el componente antes de agrupar.
    uf=Conjuntos(len(reps));lookup={tuple(sorted((r['i'],r['j']))):r for r in pares.to_dict('records')}
    for r in lookup.values():
        if r['Arista_manifestacion_segura']:uf.union(r['i'],r['j'])
    manifests=[];bridges=[]
    for comp in uf.grupos():
        dois={reps[i]['dn'] for i in comp}-{''}
        if len(dois)>1:
            anchored=defaultdict(list)
            for i in comp:
                k=reps[i]['dn'] or reps[i]['rid']
                anchored[k].append(i)
            manifests.extend(anchored.values());bridges.append(comp)
        elif len(comp)>1 and not dois and not all(lookup.get(tuple(sorted((i,j))),{}).get('Arista_manifestacion_segura',False) for i,j in combinations(comp,2)):
            manifests.extend([[i] for i in comp]);bridges.append(comp)
        else:
            # Si tiene DOI único, ningún par puede tener títulos incompatibles.
            if len({reps[i]['tn'] for i in comp})>1:
                manifests.extend([[i] for i in comp]);bridges.append(comp)
            else:manifests.append(comp)
    manifests=sorted(manifests,key=lambda g:min(reps[i]['first'] for i in g))
    mid={i:id_grupo('M',g,reps) for g in manifests for i in g}
    # Obra confirmada por clique de evidencia fuerte. Un componente NO es
    # automáticamente una obra: si hay enlace transitivo dudoso se revisa entero.
    wu=Conjuntos(len(reps))
    for g in manifests:
        for i in g[1:]:wu.union(g[0],i)
    for r in lookup.values():
        if r['Decision_obra']=='MISMA_OBRA':wu.union(r['i'],r['j'])
    works=[];uncertain=[]
    for comp in wu.grupos():
        ok=all(mid[i]==mid[j] or lookup.get(tuple(sorted((i,j))),{}).get('Decision_obra')=='MISMA_OBRA' for i,j in combinations(comp,2))
        if ok:works.append(comp)
        else:
            uncertain.append(comp)
            for g in manifests:
                part=[i for i in g if i in set(comp)]
                if part:works.append(part)
    wid={i:id_grupo('O',g,reps) for g in works for i in g}
    return manifests,works,mid,wid,bridges,uncertain


def filas_de(inds,reps):
    return sorted(set(v for i in inds for v in reps[i]['row_ids']))


def afiliaciones_consistentes(sub):
    sets=[]
    for r in sub.to_dict('records'):
        spec=frozenset(a for a in [r['Afiliacion1'],r['Afiliacion2']] if a and a!=GENERICA)
        if spec:sets.append(spec)
    union=set().union(*sets) if sets else set()
    if len(union)>2:return [],False,'MAS_DE_DOS_AFILIACIONES'
    if union and not any(s==union for s in sets):return [],False,'AFILIACIONES_ESPECIFICAS_SIN_ANCLA_CONJUNTA'
    if union:
        ordered=unicos(a for r in sub.to_dict('records') for a in [r['Afiliacion1'],r['Afiliacion2']] if a in union)
        return ordered,True,'ESPECIFICA_SIN_CONTRADICCION'
    return [GENERICA] if GENERICA in set(sub.Afiliacion1)|set(sub.Afiliacion2) else [],True,'GENERICA'


def conflictos_grupo(inds,reps,df,level='MANIFESTACION'):
    sub=df.iloc[[i-1 for i in filas_de(inds,reps)]]
    conflicts=[]
    for c in ['Año','Area']:
        if len(set(sub[c])- {''})>1:conflicts.append(c)
    if level=='MANIFESTACION':
        if len(set(sub.Doi)-{''})>1:conflicts.append('Doi')
        if len({norm_url(u) for u in sub.URL if u})>1:conflicts.append('URL')
        if len({norm_titulo(t) for t in sub.Titulo})>1:conflicts.append('Titulo')
        for c in ['ISBN','ISSN']:
            vals={frozenset(norm_id(t) for t in tokens_celda(v)) for v in sub[c] if v}
            if any(not a&b for a,b in combinations(vals,2)):conflicts.append(c)
    if not abstract_mayor(sub.Abstract)[1]:conflicts.append('Abstract')
    aff_conf=[]
    for autor,rows in sub.groupby('Autor_norm',sort=False):
        if not afiliaciones_consistentes(rows)[1]:aff_conf.append(autor)
    return conflicts,aff_conf

# ============================================================
# PLAN REVISABLE. Columnas manuales separadas de los datos de entrada.
# ============================================================
PLAN_MANUAL=['Obra_manual','Manifestacion_manual','Comentario_manual']
REV_MANUAL=['Decision_manual','row_ids_evidencia','Comentario_manual']
REV_COLS=['Caso_ID','Tipo','Nivel','Objetivo_ID','Autor_norm','Campo','row_ids_originales',
          'Representaciones','Valores_en_conflicto','Motivo','Acciones_permitidas',
          'Decision_manual','row_ids_evidencia','Comentario_manual','SHA256_entrada']


def crear_plan(reps,mid,wid,df,sha):
    rows=[]
    for i,r in enumerate(reps):
        rows.append({'Representacion_ID':r['rid'],'row_ids_originales':' | '.join(map(str,r['row_ids'])),
            'indices_originales':' | '.join(df.iloc[[k-1 for k in r['row_ids']]].indice),
            **{c:r[c] for c in BIB},'Autores':' | '.join(sorted(r['authors'])),
            'Obra_propuesta':wid[i],'Manifestacion_propuesta':mid[i],
            'Obra_manual':'','Manifestacion_manual':'','Comentario_manual':'','SHA256_entrada':sha})
    return pd.DataFrame(rows)


def incorporar_ediciones(actual,anterior,clave,editables,sha):
    if anterior is None:return actual
    if clave not in anterior:raise ValueError(f'El archivo de revisión anterior no tiene {clave}. NO usar revisiones históricas de la fase antigua.')
    if anterior[clave].duplicated().any():raise ValueError(f'{clave} duplicado en decisiones.')
    if not set(editables)<=set(anterior):raise ValueError('Esquema de decisiones no válido para esta versión.')
    has_edits=anterior[editables].ne('').any(axis=1)
    if not has_edits.any():return actual
    rows=anterior.loc[has_edits]
    if 'SHA256_entrada' not in rows or not rows.SHA256_entrada.eq(sha).all():
        raise ValueError('Las decisiones pertenecen a otra huella de entrada. No se trasladan silenciosamente.')
    if not set(rows[clave])<=set(actual[clave]):
        raise ValueError('Hay decisiones cuyo caso ya no existe en el plan actual. Resguardar y revisar antes de regenerar.')
    old=rows.set_index(clave);out=actual.copy()
    for idx,r in out.iterrows():
        if r[clave] in old.index:
            orig=old.loc[r[clave]]
            # Los campos de evidencia no se editan para inventar valores.
            stable=[c for c in actual if c not in editables and c in anterior and c not in {clave,'Obra_propuesta','Manifestacion_propuesta'}]
            for c in stable:
                if str(orig[c])!=str(r[c]):raise ValueError(f'Dato protegido modificado en {r[clave]} / {c}. Editar solo campos manuales.')
            for c in editables:out.at[idx,c]=orig[c]
    return out


def mapa_plan(plan,reps):
    ix=plan.set_index('Representacion_ID');mid={};wid={}
    for i,r in enumerate(reps):
        q=ix.loc[r['rid']]
        if (q.Obra_manual or q.Manifestacion_manual) and not q.Comentario_manual.strip():
            raise ValueError(f'Falta comentario de cambio de identidad en {r["rid"]}')
        mid[i]=q.Manifestacion_manual.strip() or q.Manifestacion_propuesta
        wid[i]=q.Obra_manual.strip() or q.Obra_propuesta
    mg=defaultdict(list);wg=defaultdict(list)
    for i in range(len(reps)):mg[mid[i]].append(i);wg[wid[i]].append(i)
    for m,inds in mg.items():
        if len({wid[i] for i in inds})!=1:raise ValueError(f'{m}: una manifestación no puede pertenecer a dos obras. Ajustar Obra_manual también.')
    return dict(mg),dict(wg),mid,wid


def opciones(df,ids,campo,autor=''):
    sub=df.iloc[[i-1 for i in ids]].copy();sub['_row']=ids
    if autor:sub=sub.loc[sub.Autor_norm.eq(autor)]
    out=[]
    if campo=='Afiliaciones':
        groups=defaultdict(list)
        for r in sub.to_dict('records'):groups[(r['Afiliacion1'],r['Afiliacion2'])].append(r['_row'])
        for value,rids in groups.items():out.append({'valor':list(value),'row_ids':rids})
    else:
        for value,g in sub.groupby(campo,sort=False,dropna=False):out.append({'valor':value,'row_ids':g['_row'].tolist()})
    return js(out)


def crear_casos(df,reps,pares,mg,wg,mid,wid,sha):
    cases=[]
    def add(tipo,nivel,obj,ids,campo,motivo,acciones,autor='',values='',reprs=None):
        ids=sorted(set(ids));key=[tipo,nivel,obj,campo,autor,ids]
        cases.append({'Caso_ID':'C_'+huella_objeto(key)[:16].upper(),'Tipo':tipo,'Nivel':nivel,'Objetivo_ID':obj,
           'Autor_norm':autor,'Campo':campo,'row_ids_originales':' | '.join(map(str,ids)),
           'Representaciones':' | '.join(sorted(reprs or [])),
           'Valores_en_conflicto':values if values else (opciones(df,ids,campo,autor) if campo in BIB+['Afiliaciones'] else ''),
           'Motivo':motivo,'Acciones_permitidas':'; '.join(acciones),
           'Decision_manual':'','row_ids_evidencia':'','Comentario_manual':'','SHA256_entrada':sha})
    # La evidencia sigue por pares, pero la revisión se agrupa por pareja de
    # manifestaciones propuestas para no preguntar lo mismo por cada metadato.
    ig=defaultdict(list)
    for p in pares.to_dict('records'):
        if not p['Candidato_operativo']:continue
        i,j=p['i'],p['j']
        issue=(p['Decision_obra']=='REVISION' or p['Decision_manifestacion']=='REVISION'
               or 'POSIBLE_ERROR' in p['Alertas']
               or (p['Arista_manifestacion_segura'] and mid[i]!=mid[j])
               or (p['Decision_obra']=='MISMA_OBRA' and wid[i]!=wid[j]))
        if issue:ig[tuple(sorted((mid[i],mid[j])))].append(p)
    for mk,ps in sorted(ig.items()):
        inds=sorted(set(i for mm in mk for i in mg[mm]))
        vals=js({'pares':[p['Par_ID'] for p in ps],
                 'manifestaciones':{mm:{'row_ids':filas_de(mg[mm],reps),
                     'Titulos':unicos(reps[i]['Titulo'] for i in mg[mm]),
                     'DOIs':unicos(reps[i]['Doi'] for i in mg[mm]),
                     'Obras':sorted(set(wid[i] for i in mg[mm]))} for mm in set(mk)}})
        why='; '.join(sorted(set(p['Regla_match'] for p in ps)))
        alerts='; '.join(sorted(set(p['Alertas'] for p in ps if p['Alertas'])))
        obj='I_'+huella_objeto(list(mk))[:16].upper()
        add('IDENTIDAD','PAR_MANIFESTACIONES',obj,filas_de(inds,reps),'Identidad',
            why+'; '+alerts,['VALIDAR_PLAN'],values=vals,reprs=[reps[i]['rid'] for i in inds])
    for mid_,inds in mg.items():
        ids=filas_de(inds,reps);conf,ac=conflictos_grupo(inds,reps,df)
        for c in conf:
            actions=['CORREGIR_DOI_INTERNO'] if c=='Doi' else ['SELECCIONAR','COMBINAR'] if c in {'ISBN','ISSN'} else ['SELECCIONAR']
            add('CAMPO','MANIFESTACION',mid_,ids,c,'Valores distintos; no se decide por mayoría ni por fuente.',actions)
        for autor in ac:
            add('AFILIACION','MANIFESTACION',mid_,ids,'Afiliaciones','Específicas distintas sin una fila que avale conjuntamente el par, o más de dos.',
                ['SELECCIONAR','COMBINAR'],autor=autor)
        if sub_no_ids(df,ids):
            add('SIN_IDENTIFICADORES','MANIFESTACION',mid_,ids,'Identidad',
                'Todos los identificadores están vacíos. El perfil idéntico es solo una representación auxiliar, no prueba de edición.',
                ['VALIDAR_EXISTENTE'])
        # Inconsistencia interna URL DOI: no se ignora aunque perfil único.
        if any(reps[i]['doi_url_conflict'] for i in inds):
            add('IDENTIFICADORES','MANIFESTACION',mid_,ids,'URL','Una URL doi.org contiene DOI distinto de Doi. Solo evidencia interna; no corregir automáticamente.',
                ['SELECCIONAR','VALIDAR_EXISTENTE'])
    for oid,inds in wg.items():
        mids={mid[i] for i in inds}
        if len(mids)<=1:continue
        ids=filas_de(inds,reps);sub=df.iloc[[v-1 for v in ids]]
        risk=any(reps[i]['version'] for i in inds)
        if risk:
            add('VERSION','OBRA',oid,ids,'Contenido','La obra enlaza indicios de preprint/versión. No armonizar contenido entre versiones sin aprobación.',
                ['ARMONIZAR_CONTENIDO','NO_ARMONIZAR_VERSIONES'])
        conf,ac=conflictos_grupo(inds,reps,df,'OBRA')
        for c in conf:
            add('CAMPO','OBRA',oid,ids,c,'Conflicto entre manifestaciones de una obra propuesta. No decidir por mayoría.',
                (['SELECCIONAR'] if c=='Area' else ['SELECCIONAR','MANTENER_POR_VERSION']))
        for autor in ac:
            add('AFILIACION','OBRA',oid,ids,'Afiliaciones','No propagar entre afiliaciones UNAM específicas contradictorias.',
                ['MANTENER_POR_MANIFESTACION'],autor=autor)
        # Precisar afiliación entre manifestaciones requiere aprobación específica,
        # además de la identidad de obra. No trasladar adscripciones históricas.
        for mid_ in sorted(mids):
            target=filas_de(mg[mid_],reps)
            tdf=df.iloc[[v-1 for v in target]]
            for autor,rows in tdf.groupby('Autor_norm',sort=False):
                current,ok,_=afiliaciones_consistentes(rows)
                donor=sub.loc[sub.Autor_norm.eq(autor)]
                complete,cok,_=afiliaciones_consistentes(donor)
                if ok and cok and current==[GENERICA] and complete and complete!=[GENERICA]:
                    add('AFILIACION_GENERICA','MANIFESTACION',mid_,ids,'Afiliaciones',
                        'Dependencia del mismo autor en otra manifestación de la obra. Requiere confirmar aplicabilidad a la versión destino.',
                        ['PROPAGAR','NO_PROPAGAR'],autor=autor)
        allauthors=set(sub.Autor_norm)
        for mid_ in sorted(mids):
            target=filas_de(mg[mid_],reps);present=set(df.iloc[[v-1 for v in target]].Autor_norm)
            for autor in sorted(allauthors-present):
                donor=[v for v in ids if df.iloc[v-1].Autor_norm==autor]
                add('AUTOR_FALTANTE','MANIFESTACION',mid_,ids,'Autor_norm',
                    'Diferencia observada, no omisión demostrada. Verificar misma autoría y ausencia de cambio de versión; donantes: '+ ' | '.join(map(str,donor)),
                    ['PROPAGAR','NO_PROPAGAR'],autor=autor)
    return pd.DataFrame(cases,columns=REV_COLS).fillna('')


def potenciales_internos(df,reps,mid,wid):
    # Diagnóstico amplio de POTENCIALES, no operaciones aplicadas. Se busca dentro
    # del título normalizado, con banderas de obra/manifestación propuestas.
    out=[];bytitle=defaultdict(list)
    # Familias SOLO para búsqueda de oportunidades: título exacto O DOI igual.
    # No se usan para ejecutar propagación ni para aprobar una obra.
    uf=Conjuntos(len(reps));seen_t={};seen_d={};seen_w={}
    for i,r in enumerate(reps):
        for val,mapping in [(r['tn'],seen_t),(r['dn'],seen_d),(wid[i],seen_w)]:
            if not val:continue
            if val in mapping:uf.union(i,mapping[val])
            else:mapping[val]=i
    for g in uf.grupos():bytitle[min(g)]=g
    for tn,inds in bytitle.items():
        ids=filas_de(inds,reps);sub=df.iloc[[v-1 for v in ids]].copy();sub['_row']=ids
        byman=defaultdict(list)
        for i in inds:byman[mid[i]].append(i)
        authors=set(sub.Autor_norm)
        for m,gg in byman.items():
            target=filas_de(gg,reps);ta=set(df.iloc[[v-1 for v in target]].Autor_norm)
            for autor in sorted(authors-ta):
                donors=sub.loc[sub.Autor_norm.eq(autor),'_row'].tolist()
                donorinds=[i for i in inds if autor in reps[i]['authors']]
                samework=all(wid[i]==wid[gg[0]] for i in donorinds)
                version=any(reps[i]['version'] for i in inds)
                out.append({'Tipo':'AUTOR_POTENCIAL','Titulo':reps[gg[0]]['Titulo'],'Autor_norm':autor,
                    'Manifestacion_destino':m,'row_ids_destino':' | '.join(map(str,target)),
                    'row_ids_donantes':' | '.join(map(str,donors)),'Campo':'Autor_norm','Valor_propuesto':autor,
                    'Estado':'REVISION_VERSION' if version else 'PROPUESTA_MISMA_OBRA' if samework else 'REQUIERE_CONFIRMAR_OBRA',
                    'Comentario':'Familia candidata por título exacto o DOI igual. No agregar solo porque exista en el archivo; diferencia de conjuntos no prueba omisión.'})
        for autor,g in sub.groupby('Autor_norm',sort=False):
            gen=g.loc[g.Afiliacion1.eq(GENERICA)&g.Afiliacion2.eq('')]
            spec=g.loc[~g.Afiliacion1.eq(GENERICA)|g.Afiliacion2.ne('')]
            if gen.empty or spec.empty:continue
            aff,ok,why=afiliaciones_consistentes(spec)
            for rr in gen.to_dict('records'):
                i=next(i for i in inds if rr['_row'] in reps[i]['row_ids'])
                di=[i for i in inds if set(reps[i]['row_ids'])&set(spec['_row'])]
                within=any(mid[k]==mid[i] for k in di)
                same=all(wid[k]==wid[i] for k in di)
                out.append({'Tipo':'AFILIACION_GENERICA','Titulo':rr['Titulo'],'Autor_norm':autor,
                   'Manifestacion_destino':mid[i],'row_ids_destino':str(rr['_row']),
                   'row_ids_donantes':' | '.join(map(str,spec['_row'].tolist())),
                   'Campo':'Afiliacion1/Afiliacion2','Valor_propuesto':'; '.join(aff) if ok else '',
                   'Estado':'CONFLICTO_ESPECIFICAS' if not ok else 'DENTRO_MANIFESTACION' if within else 'ENTRE_MANIFESTACIONES_REVISAR' if same else 'REQUIERE_CONFIRMAR_OBRA',
                   'Comentario':why+'; nunca usar la afiliación de otro autor.'})
    return pd.DataFrame(out,columns=['Tipo','Titulo','Autor_norm','Manifestacion_destino','row_ids_destino',
       'row_ids_donantes','Campo','Valor_propuesto','Estado','Comentario'])


def sub_no_ids(df,ids):
    return bool(df.iloc[[v-1 for v in ids]][['Doi','ISBN','ISSN','URL']].eq('').all(axis=None))

# ============================================================
# DIAGNÓSTICO A-Q Y ARCHIVOS SIN MODIFICAR LA BASE
# ============================================================

def diagnostico(df,reps,pares,cal,mg,wg,mid,wid,rev,pot,sha,vecinos):
    rows=[]
    def add(b,m,v,nota='',unidad=''):
        rows.append({'Apartado':b,'Metrica':m,'Valor':v,'Unidad':unidad,'Detalle':nota})
    add('A','filas_entrada',len(df),'Ninguna eliminada ni fusionada en DIAGNOSTICO.','filas')
    add('A','columnas_trabajo_sin_Fuente_origen',len(df.columns),'Fuente_origen eliminada antes de generar perfiles/features.','columnas')
    add('A','sha256_entrada',sha)
    add('B','titulos_unicos_exactos',df.Titulo.nunique(),unidad='cadenas')
    tn=df.Titulo.map(norm_titulo);add('B','titulos_unicos_normalizados_temporales',tn.nunique(),unidad='cadenas')
    vc=df.Titulo.value_counts();add('C','grupos_titulo_exacto_repetido',int(vc.gt(1).sum()),unidad='grupos')
    add('C','filas_en_titulos_exactos_repetidos',int(vc[vc.gt(1)].sum()),unidad='filas')
    dc=df.loc[df.Doi.ne(''),'Doi'].value_counts()
    add('D','dois_no_vacios_distintos',len(dc),unidad='DOI')
    add('D','grupos_doi_repetido',int(dc.gt(1).sum()),unidad='grupos')
    add('D','filas_en_doi_repetido',int(dc[dc.gt(1)].sum()),unidad='filas')
    spread=df.loc[df.Doi.ne('')].groupby('Doi')['indice'].nunique()
    add('E','dois_en_varios_indices_historicos',int(spread.gt(1).sum()),'Indices históricos son etiquetas de auditoría, no llaves.','DOI')
    add('E','dois_con_varios_titulos_exactos',int(df.loc[df.Doi.ne('')].groupby('Doi').Titulo.nunique().gt(1).sum()),unidad='DOI')
    for label,keys in [('exacto',df.Titulo),('normalizado',tn)]:
        gr=df.assign(_key=keys).groupby('_key',sort=False)
        mult=gr.Doi.agg(lambda s:len(set(s)-{''}))
        add('F','titulos_'+label+'_con_doi_distintos',int(mult.gt(1).sum()),unidad='grupos')
        incomplete=0
        for k,g in gr:
            if len(g)>1 and g[['Doi','ISBN','ISSN','URL']].eq('').any(axis=None):incomplete+=1
        add('G','titulos_'+label+'_repetidos_con_bibliografia_parcial',incomplete,'Incluye campos que pueden no aplicar: no supone que todo artículo deba tener ISBN e ISSN.','grupos')
    for c in ['ISBN','ISSN','Doi','URL','Año','Keywords','Abstract']:
        n=int(df[c].eq('').sum());add('H','vacios_'+c,n,f'{n/len(df):.6%}', 'filas')
    patterns=Counter(tuple(row) for row in df[['Doi','ISBN','ISSN','URL']].eq('').itertuples(index=False,name=None))
    for pat,n in sorted(patterns.items()):
        add('H','patron_identificadores',n,js(dict(zip(['Doi_vacio','ISBN_vacio','ISSN_vacio','URL_vacia'],pat))),'filas')
    add('H','todos_identificadores_vacios',int(df[['Doi','ISBN','ISSN','URL']].eq('').all(axis=1).sum()),unidad='filas')
    add('I','representaciones_auxiliares',len(reps),'Perfiles de 10 campos bibliográficos; conservan la lista completa de filas. No son aún manifestaciones.','perfiles')
    add('I','pares_comparados',len(pares),'Incluye controles de calibración.','pares')
    add('I','pares_operativos',int(pares.Candidato_operativo.sum()),unidad='pares')
    add('I','vecinos_qgram_por_titulo',vecinos,'Presupuesto de recuperación, no threshold de aprobación. No usa Fuente_origen.')
    add('I','obras_propuestas',len(wg),'Agrupaciones auxiliares: NO recuento definitivo de obras.','grupos')
    for st,n in pares.loc[pares.Candidato_operativo.eq(1)].Decision_obra.value_counts().items():add('I',st,int(n),unidad='pares')
    add('J','manifestaciones_propuestas',len(mg),'Antes de aprobación y de resolver conflictos; no se han asignado índices finales.','grupos')
    for st,n in pares.loc[pares.Candidato_operativo.eq(1)].Decision_manifestacion.value_counts().items():add('J',st,int(n),unidad='pares')
    add('K','potenciales_autor_manifestacion',int(pot.Tipo.eq('AUTOR_POTENCIAL').sum()),'No equivalen a omisiones comprobadas; cero autores agregados en diagnóstico.','propuestas')
    add('L','filas_unam_generica',int((df.Afiliacion1.eq(GENERICA)&df.Afiliacion2.eq('')).sum()),unidad='filas')
    for st,n in pot.loc[pot.Tipo.eq('AFILIACION_GENERICA')].Estado.value_counts().items():add('L',st,int(n),unidad='filas')
    for b,c in [('M','Area'),('N','Año')]:
        g=df.assign(_t=tn).groupby('_t')[c].agg(lambda s:len(set(s)-{''}))
        add(b,'titulos_normalizados_con_conflicto_'+c,int(g.gt(1).sum()),unidad='grupos')
        add(b,'manifestaciones_propuestas_con_conflicto_'+c,sum(c in conflictos_grupo(v,reps,df)[0] for v in mg.values()),unidad='grupos')
    add('O','metodo_pesos','LOG_RAZON_VEROSIMILITUD_EMPIRICA',
        'Estados por campo + suavizado 0.5. Ponderación por familia de títulos, partición por familia DOI/título. Ausentes pesan cero. Correlaciones entre campos impiden interpretar como probabilidad. Ver calibracion_deduplicacion.csv.')
    add('O','reglas_duras','PREVIAS_AL_SCORE','DOI diferentes no se colapsan; DOI igual/título compatible identifica pero no resuelve conflictos de campo.')
    for r in cal.loc[cal.Tipo.eq('UMBRAL_PROPUESTO')].to_dict('records'):
        add('P',r['Nivel']+'_'+r['Estado'],r.get('Valor',''),r['Detalle'],'score log2 (NO probabilidad)')
    add('P','calibracion','EXPLORATORIA_NO_GROUND_TRUTH','Los ejemplos internos están definidos con las mismas features: no son una evaluación independiente, especialmente para pares sin DOI o entre versiones.')
    blocked_m=set(rev.loc[rev.Nivel.eq('MANIFESTACION'),'Objetivo_ID'])
    blocked_rep=set(v for s in rev.loc[rev.Tipo.eq('IDENTIDAD'),'Representaciones'] for v in s.split(' | ') if v)
    safe=[];uncertain=[];single=[]
    for m,g in mg.items():
        ids=filas_de(g,reps);sub=df.iloc[[v-1 for v in ids]]
        blocked=m in blocked_m or any(reps[i]['rid'] in blocked_rep for i in g)
        if blocked:uncertain.append(m)
        elif len(sub)>sub.Autor_norm.nunique():safe.append(m)
        else:single.append(m)
    distinct=pares.loc[pares.Candidato_operativo.eq(1)&pares.Decision_manifestacion.eq('DISTINTA_MANIFESTACION')&~pares.Decision_obra.eq('OBRA_DISTINTA')]
    add('Q','AUTO_FUSION_estimada',len(safe),'Manifestaciones propuestas sin casos pendientes, con relaciones repetidas. No ejecutadas.','grupos')
    add('Q','REVISION_manifestaciones_propuestas',len(uncertain),'Afectadas por identidad o conflictos propios.','grupos')
    add('Q','SIN_FUSION_NECESARIA',len(single),'No contienen autores repetidos ni conflictos detectados.','grupos')
    add('Q','DISTINTA_MANIFESTACION_pares_misma_obra_o_posible',len(distinct),'No se suma a los grupos anteriores: son pares, no manifestaciones.','pares')
    add('Q','casos_revision_totales',len(rev),'Tipos/campos pueden coincidir sobre las mismas filas.','casos')
    for t,n in rev.Tipo.value_counts().items():add('Q','revision_'+t,int(n),unidad='casos')
    add('Q','relaciones_proyectadas_sin_agregar_autores',sum(df.iloc[[v-1 for v in filas_de(g,reps)]].Autor_norm.nunique() for g in mg.values()),
        'Proyección de la partición propuesta. NO resultado aprobado ni archivo final.','relaciones')
    add('CONTROL','indices_finales_asignados',0)
    add('CONTROL','filas_eliminadas_o_fusionadas_en_diagnostico',0)
    add('CONTROL','consultas_de_internet',0)
    add('CONTROL','datos_externos_utilizados',0)
    add('Q','DISTINTA_MANIFESTACION_misma_obra_fuerte',int((distinct.Decision_obra=='MISMA_OBRA').sum()),'Pares con obra fuerte; aún se controlan versiones y conflictos.','pares')
    add('Q','DISTINTA_MANIFESTACION_obra_por_revisar',int((distinct.Decision_obra=='REVISION').sum()),'No confirma que sean la misma obra.','pares')
    add('CONTROL','version_pandas',pd.__version__)
    add('CONTROL','version_numpy',np.__version__)
    for r in rows:r['Tipo']='RESUMEN'
    # Detalle local de los grupos solicitados A-Q, en EL MISMO archivo diagnóstico.
    logical=df.copy();logical['_row']=range(1,len(df)+1);logical['_titulo_n']=tn
    def detail(b,metric,key,g):
        rows.append({'Apartado':b,'Metrica':metric,'Valor':len(g),'Unidad':'filas','Tipo':'GRUPO',
             'Clave':key,'Titulo':' || '.join(unicos(g.Titulo)),'Doi':' | '.join(unicos(g.Doi)),
             'row_ids_originales':' | '.join(map(str,g['_row'])),
             'indices_originales':' | '.join(g.indice),
             'Detalle':js({'Años':unicos(g['Año']),'Areas':unicos(g.Area),'Autores':unicos(g.Autor_norm),
                  'DOI_vacios':int(g.Doi.eq('').sum()),'ISBN_vacios':int(g.ISBN.eq('').sum()),
                  'ISSN_vacios':int(g.ISSN.eq('').sum()),'URL_vacias':int(g.URL.eq('').sum())})})
    for key,g in logical.groupby('Titulo',sort=False):
        if len(g)>1:detail('C','DETALLE_TITULO_EXACTO_REPETIDO',key,g)
    for key,g in logical.loc[logical.Doi.ne('')].groupby('Doi',sort=False):
        if len(g)>1:detail('D','DETALLE_DOI_REPETIDO',key,g)
        if g.indice.nunique()>1:detail('E','DETALLE_DOI_VARIOS_INDICES',key,g)
    for key,g in logical.groupby('_titulo_n',sort=False):
        if len(set(g.Doi)-{''})>1:detail('F','DETALLE_TITULO_DOI_DISTINTOS',key,g)
        if len(g)>1 and g[['Doi','ISBN','ISSN','URL']].eq('').any(axis=None):detail('G','DETALLE_BIBLIOGRAFIA_PARCIAL',key,g)
        if g.Area.nunique()>1:detail('M','DETALLE_CONFLICTO_AREA',key,g)
        if len(set(g['Año'])-{''})>1:detail('N','DETALLE_CONFLICTO_ANIO',key,g)
    return pd.DataFrame(rows)

# ============================================================
# FUSIÓN FINAL (solo con aprobación y TODAS las revisiones cerradas)
# ============================================================

def parse_ids(s):
    if not str(s).strip():return []
    parts=[p.strip() for p in str(s).split('|')]
    if any(not re.fullmatch(r'\d+',p) for p in parts):raise ValueError('Usar row_ids enteros separados por |; no índices históricos ni decimales.')
    return sorted(set(int(p) for p in parts))


def decisiones_validadas(rev,df,sha):
    if not rev.empty and not rev.SHA256_entrada.eq(sha).all():raise ValueError('Huella incorrecta en revisión.')
    missing=rev.loc[rev.Decision_manual.eq('')]
    if len(missing):raise ValueError(f'FUSIÓN BLOQUEADA: {len(missing)} casos siguen sin decisión. No se genera una base parcial como definitiva.')
    out={}
    for r in rev.to_dict('records'):
        if r['Decision_manual'] not in tokens_celda(r['Acciones_permitidas']):
            raise ValueError(f'{r["Caso_ID"]}: decisión no permitida: {r["Decision_manual"]}')
        if not r['Comentario_manual'].strip():raise ValueError(f'{r["Caso_ID"]}: falta la justificación interna.')
        evidence=parse_ids(r['row_ids_evidencia'])
        scope=set(parse_ids(r['row_ids_originales']))
        if not evidence or not set(evidence)<=scope:raise ValueError(f'{r["Caso_ID"]}: evidencia vacía o fuera del contexto permitido.')
        if min(evidence)<1 or max(evidence)>len(df):raise ValueError('row_id fuera de la entrada.')
        key=(r['Tipo'],r['Nivel'],r['Objetivo_ID'],r['Autor_norm'],r['Campo'])
        if key in out:raise ValueError('Decisión contextual duplicada.')
        out[key]={**r,'evidence':evidence}
    return out


def fusion_final(df,reps,pares,plan,rev,sha):
    mg,wg,mid,wid=mapa_plan(plan,reps)
    dec=decisiones_validadas(rev,df,sha)
    rows_by_m={m:filas_de(g,reps) for m,g in mg.items()}
    rows_by_w={w:filas_de(g,reps) for w,g in wg.items()}
    work_of_m={m:wid[g[0]] for m,g in mg.items()}
    field_audit=[]
    def get(tipo,nivel,obj,campo,autor=''):
        return dec.get((tipo,nivel,obj,autor,campo))
    def select_field(ids,c,nivel,obj):
        sub=df.iloc[[v-1 for v in ids]]
        vals=unicos(sub[c]);decision=get('CAMPO',nivel,obj,c)
        if decision:
            action=decision['Decision_manual'];ev=decision['evidence']
            if action=='MANTENER_POR_VERSION':return None,[],action
            chosen=unicos(df.iloc[[v-1 for v in ev]][c])
            if c in {'ISBN','ISSN'} and action=='COMBINAR':
                value='; '.join(unicos((v for value in chosen for v in tokens_celda(value)),norm_id))
            else:
                if len(chosen)!=1:raise ValueError(f'{decision["Caso_ID"]}: seleccionar evidencia de un único valor no vacío.')
                value=chosen[0]
            if c=='Doi' and action!='CORREGIR_DOI_INTERNO':raise ValueError('No corregir DOI por una selección genérica.')
            return value,ev,'MANUAL_'+action
        if c in {'ISBN','ISSN','Keywords'}:
            key=norm_id if c in {'ISBN','ISSN'} else lambda s:texto(s).casefold()
            value='; '.join(unicos((t for v in sub[c] for t in tokens_celda(v)),key))
            return value,[i for i in ids if df.iloc[i-1][c]],'UNION_DE_COMPONENTES_EXISTENTES'
        if not vals:return '',[],'TODOS_VACIOS'
        if c=='Abstract':
            value,ok,why=abstract_mayor(vals)
            if not ok:raise ValueError(f'{nivel}/{obj}: Abstract distinto sin selección.')
            return value,[i for i in ids if df.iloc[i-1][c]==value],why
        key=norm_url if c=='URL' else norm_titulo if c=='Titulo' else lambda s:s
        if len({key(v) for v in vals})>1:raise ValueError(f'{nivel}/{obj}: conflicto {c} sin resolver.')
        # Elige un valor que YA está en el archivo. No construye otra URL/título.
        value=(min(vals,key=lambda v:(not v.startswith('https://'),len(v),v)) if c=='URL' else vals[0])
        return value,[i for i in ids if df.iloc[i-1][c]==value],'VALOR_EXISTENTE_EQUIVALENTE'
    articles={};provenance={};harmonize={}
    for w,inds in wg.items():
        ds=get('VERSION','OBRA',w,'Contenido')
        harmonize[w]=not ds or ds['Decision_manual']=='ARMONIZAR_CONTENIDO'
    for m,ids in rows_by_m.items():
        art={};prov={}
        for c in BIB:
            if c=='SubArea':art[c]='';prov[c]=([], 'SUBAREA_VACIA');continue
            value,donors,method=select_field(ids,c,'MANIFESTACION',m)
            art[c]=value;prov[c]=(donors,method)
        check=get('IDENTIFICADORES','MANIFESTACION',m,'URL')
        if check and check['Decision_manual']=='SELECCIONAR':
            vals=unicos(df.iloc[[v-1 for v in check['evidence']]].URL)
            if len(vals)!=1:raise ValueError('Escoger una URL existente para resolver contradicción DOI-URL.')
            art['URL']=vals[0];prov['URL']=(check['evidence'],'MANUAL_IDENTIFICADOR_URL')
        if doi_url(art['URL']) and art['Doi'] and doi_url(art['URL'])!=art['Doi']:
            raise ValueError(f'{m}: persiste contradicción DOI vs URL doi.org. No basta VALIDAR_EXISTENTE.')
        articles[m]=art;provenance[m]=prov
    # Campos de obra: año/área coherentes, Keywords unión y Abstract único/no concatenado.
    exceptions=[]
    for w,inds in wg.items():
        ms=sorted({mid[i] for i in inds},key=lambda m:min(rows_by_m[m]));ids=rows_by_w[w]
        if len(ms)<2:continue
        for c in ['Año','Area','Keywords','Abstract']:
            if c in {'Keywords','Abstract'} and not harmonize[w]:continue
            decision=get('CAMPO','OBRA',w,c)
            if decision and decision['Decision_manual']=='MANTENER_POR_VERSION':
                exceptions.append((w,c,decision['Caso_ID']));continue
            value,donors,method=select_field(ids,c,'OBRA',w)
            for m in ms:
                articles[m][c]=value;provenance[m][c]=(donors,'OBRA_'+method)
    # Orden de índices: primera aparición original de CADA manifestación; desempate ID.
    ordered=sorted(mg,key=lambda m:(min(rows_by_m[m]),m))
    newindex={m:str(i+1) for i,m in enumerate(ordered)}
    out=[];lineage=[];added=0
    for m in ordered:
        ids=rows_by_m[m];w=work_of_m[m];wids=rows_by_w[w]
        sub=df.iloc[[v-1 for v in ids]]
        original_authors=unicos(sub.Autor_norm)
        authors=original_authors.copy()
        for a in sorted(set(df.iloc[[v-1 for v in wids]].Autor_norm)-set(authors)):
            ac=get('AUTOR_FALTANTE','MANIFESTACION',m,'Autor_norm',a)
            if ac and ac['Decision_manual']=='PROPAGAR':authors.append(a)
        for a in authors:
            direct=[v for v in ids if df.iloc[v-1].Autor_norm==a]
            ac=get('AUTOR_FALTANTE','MANIFESTACION',m,'Autor_norm',a)
            if not direct:
                added+=1;arows=ac['evidence']
                if any(df.iloc[v-1].Autor_norm!=a for v in arows):raise ValueError('Donante del autor es otra persona.')
            else:arows=direct
            local=df.iloc[[v-1 for v in arows]]
            aff,ok,why=afiliaciones_consistentes(local)
            afdec=get('AFILIACION','MANIFESTACION',m,'Afiliaciones',a)
            donors=arows.copy()
            if afdec:
                donors=afdec['evidence']
                if any(df.iloc[v-1].Autor_norm!=a for v in donors):raise ValueError('No copiar afiliaciones de otro autor.')
                afrows=df.iloc[[v-1 for v in donors]]
                if afdec['Decision_manual']=='SELECCIONAR':
                    pairs=set(zip(afrows.Afiliacion1,afrows.Afiliacion2))
                    if len(pairs)!=1:raise ValueError('SELECCIONAR requiere el mismo par de afiliaciones en la evidencia.')
                aff=unicos(v for row in afrows[['Afiliacion1','Afiliacion2']].itertuples(index=False,name=None) for v in row if v and v!=GENERICA)
                if not aff:aff=[GENERICA] if GENERICA in set(afrows.Afiliacion1)|set(afrows.Afiliacion2) else []
                ok=len(aff)<=2;why='MANUAL_AFILIACIONES'
            upgrade=get('AFILIACION_GENERICA','MANIFESTACION',m,'Afiliaciones',a)
            if upgrade and upgrade['Decision_manual']=='PROPAGAR':
                donors=upgrade['evidence']
                if any(df.iloc[v-1].Autor_norm!=a for v in donors):raise ValueError('Afiliación genérica: el donante es otro autor.')
                if get('AFILIACION','OBRA',w,'Afiliaciones',a):raise ValueError('No precisar genérica entre específicas contradictorias de la obra.')
                aff,ok,why=afiliaciones_consistentes(df.iloc[[v-1 for v in donors]])
                if aff==[GENERICA]:raise ValueError('La evidencia no ofrece una dependencia específica.')
                why='MANUAL_PRECISION_MISMO_AUTOR_OBRA'
            if not ok or not 1<=len(aff)<=2:raise ValueError(f'{m}/{a}: afiliaciones no resueltas; jamás truncar.')
            result={'indice':newindex[m],**articles[m],'Autor_norm':a,'Afiliacion1':aff[0],
                    'Afiliacion2':aff[1] if len(aff)==2 else ''}
            out.append({c:result[c] for c in CANON})
            propagated=[c for c in BIB if any(df.iloc[v-1][c]!=articles[m][c] for v in direct)] if direct else ['Autor_norm']
            if direct and any((df.iloc[v-1].Afiliacion1,df.iloc[v-1].Afiliacion2)!=(result['Afiliacion1'],result['Afiliacion2']) for v in direct):
                propagated+=['Afiliacion1','Afiliacion2']
            relevant=pares.loc[pares['i'].isin(mg[m])&pares['j'].isin(mg[m])&pares.Candidato_operativo.eq(1)]
            lineage.append({'nuevo_indice':newindex[m],'Obra_ID':w,'Manifestacion_ID':m,'Autor_norm':a,
                'row_ids_originales':' | '.join(map(str,direct)),
                'indices_originales':' | '.join(df.iloc[[v-1 for v in direct]].indice) if direct else '',
                'row_ids_donantes_autor_afiliacion':' | '.join(map(str,donors)),
                'Titulo':articles[m]['Titulo'],'Autores':' | '.join(authors),
                'firma_bibliografica':js({c:articles[m][c] for c in ['Doi','ISBN','ISSN','URL']}),
                'regla_match':'; '.join(sorted(set(relevant.Regla_match))) if len(relevant) else 'PERFIL_BIBLIOGRAFICO_IDENTICO_O_UNICO',
                'score_obra':float(relevant.score_obra.min()) if len(relevant) else '',
                'score_manifestacion':float(relevant.score_manifestacion.min()) if len(relevant) else '',
                'campos_propagados':'; '.join(unicos(propagated)),
                'conflictos':' | '.join(rev.loc[rev.Objetivo_ID.isin([m,w]),'Caso_ID']),
                'decision':'AUTOR_PROPAGADO_APROBADO' if not direct else 'RELACION_CONSOLIDADA' if len(direct)>1 else 'RELACION_CONSERVADA',
                'SHA256_entrada':sha})
            for c in ['Autor_norm','Afiliacion1','Afiliacion2']:
                value=result[c]
                donors_c=[v for v in arows if df.iloc[v-1].Autor_norm==a] if c=='Autor_norm' else [v for v in donors if value and value in [df.iloc[v-1].Afiliacion1,df.iloc[v-1].Afiliacion2]]
                if value and not donors_c:raise ValueError('Nombre/afiliación sin evidencia del mismo autor.')
                field_audit.append({'nuevo_indice':newindex[m],'Obra_ID':w,'Autor_norm':a,'Campo':c,
                    'Valor_final':value,'row_ids_donantes':' | '.join(map(str,donors_c)),
                    'Metodo':'AUTOR_EXISTENTE' if c=='Autor_norm' else why,'SHA256_entrada':sha})
        for c in BIB:
            donors,method=provenance[m][c];value=articles[m][c]
            # Cada escalar procede de una fila del ámbito; cada token combinado existe
            # en el ámbito permitido. Nunca IDs de manifestaciones diferentes.
            scope=wids if method.startswith('OBRA_') else ids
            if not set(donors)<=set(scope):raise ValueError('Donante fuera de ámbito obra/manifestación.')
            if c in {'ISBN','ISSN','Keywords'}:
                key=norm_id if c!='Keywords' else lambda x:texto(x).casefold()
                existing={key(t) for v in donors for t in tokens_celda(df.iloc[v-1][c])}
                if any(key(t) not in existing for t in tokens_celda(value)):raise ValueError('Componente inventado durante fusión.')
            elif c!='SubArea' and value and not any(df.iloc[v-1][c]==value for v in donors):
                raise ValueError(f'{c}: valor final inexistente entre donantes.')
            field_audit.append({'nuevo_indice':newindex[m],'Obra_ID':w,'Autor_norm':'','Campo':c,
                'Valor_final':value,'row_ids_donantes':' | '.join(map(str,donors)),
                'Metodo':method,'SHA256_entrada':sha})
    result=pd.DataFrame(out,columns=CANON)
    validar_base(result,entrada=False)
    if result.duplicated(['indice','Autor_norm']).any():raise ValueError('Autor repetido dentro de manifestación.')
    for c in BIB:
        if result.groupby('indice')[c].nunique().gt(1).any():raise ValueError('Campos bibliográficos inconsistentes dentro del índice.')
    if set(result.Autor_norm)!=set(df.Autor_norm):raise ValueError('Se perdieron o inventaron autores.')
    covered=[v for ids in rows_by_m.values() for v in ids]
    if sorted(covered)!=list(range(1,len(df)+1)):raise ValueError('Lineage original perdido o duplicado.')
    expected=sum(df.iloc[[v-1 for v in ids]].Autor_norm.nunique() for ids in rows_by_m.values())+added
    if len(result)!=expected:raise ValueError('Conteo de relaciones final inconsistente.')
    # Nunca tolerar DOI+título compatible repartido innecesariamente en varios índices.
    doi_man=defaultdict(dict)
    for m,a in articles.items():
        if a['Doi']:
            key=norm_titulo(a['Titulo'])
            if key in doi_man[a['Doi']]:raise ValueError('Mismo DOI+título en índices distintos. Resolver el plan de identidad antes de finalizar.')
            doi_man[a['Doi']][key]=m
    # Área consistente en obra. Los años distintos solo con decisión explícita.
    for w,inds in wg.items():
        ms={mid[i] for i in inds}
        if len({articles[m]['Area'] for m in ms})>1:raise ValueError('Area sigue inconsistente dentro de la obra.')
        if len({articles[m]['Año'] for m in ms}-{''})>1 and not any(q[0]==w and q[1]=='Año' for q in exceptions):
            raise ValueError('Año contradictorio sin decisión por versión.')
    return result,pd.DataFrame(lineage),pd.DataFrame(field_audit)


# ============================================================
# ESCRITURA SEGURA Y RELECTURA DE TODOS LOS CSV
# ============================================================

def texto_dataframe(df):
    return df.fillna('').astype(str).reset_index(drop=True)


def guardar_lote(tables,actualizar=False,protegidos=()):
    protected={Path(p).resolve() for p in protegidos}
    prepared=[]
    for path,frame in tables.items():
        path=Path(path)
        if path.resolve() in protected:raise ValueError('Intento de sobrescribir un archivo de entrada.')
        value=texto_dataframe(frame)
        if any(c.startswith('Unnamed') for c in value):raise ValueError('No exportar columnas Unnamed.')
        if path.exists():
            old=leer_csv(path)
            if old.equals(value):continue
            if not actualizar:raise FileExistsError(f'{path.name} ya existe con otros datos. Resguardar en Git y autorizar actualizar_archivos=True.')
        prepared.append((path,value))
    temps=[]
    try:
        for path,value in prepared:
            path.parent.mkdir(parents=True,exist_ok=True)
            temp=path.with_name(path.name+'.__tmp__')
            if temp.exists():raise FileExistsError('Existe un temporal de ejecución previa. Revisarlo antes de continuar.')
            value.to_csv(temp,index=False,encoding='utf-8-sig',lineterminator='\n',quoting=csv.QUOTE_ALL)
            temps.append((temp,path))
            check=leer_csv(temp)
            if not check.equals(value):raise ValueError(f'Falló relectura exacta de {path.name}')
            with temp.open('r',encoding='utf-8-sig',newline='') as fh:
                if next(csv.reader(fh))!=list(value.columns):raise ValueError('Encabezado físico incorrecto.')
        for temp,path in temps:temp.replace(path)
    finally:
        for temp,_ in temps:
            if temp.exists():temp.unlink()
    for path,frame in tables.items():
        if not leer_csv(path).equals(texto_dataframe(frame)):raise ValueError(f'Verificación posterior falló: {path}')


def analisis_completo(ruta,vecinos=5,plan_anterior=None,revision_anterior=None,mostrar_progreso=False):
    df,sha=cargar(ruta)
    if mostrar_progreso:print('1/6 Entrada validada:',len(df),'filas. Fuente_origen fuera de la lógica.',flush=True)
    reps=construir_representaciones(df)
    pairs,nearest=generar_candidatos(reps,vecinos)
    if mostrar_progreso:print('2/6 Comparando',len(pairs),'pares de perfiles, sin usar los índices históricos.',flush=True)
    feats=comparar(reps,pairs)
    if mostrar_progreso:print('3/6 Pesos y umbrales exploratorios con evidencia interna.',flush=True)
    feats,cal,th=calibrar(feats,reps)
    feats=clasificar_pares(feats,reps,th)
    groups,works,mid,wid,bridges,unc=agrupar_propuestas(reps,feats)
    plan=crear_plan(reps,mid,wid,df,sha)
    plan=incorporar_ediciones(plan,plan_anterior,'Representacion_ID',PLAN_MANUAL,sha)
    mg,wg,mid,wid=mapa_plan(plan,reps)
    if mostrar_progreso:print('4/6 Plan de identidad y conflictos por campo.',flush=True)
    rev=crear_casos(df,reps,feats,mg,wg,mid,wid,sha)
    rev=incorporar_ediciones(rev,revision_anterior,'Caso_ID',REV_MANUAL,sha)
    pot=potenciales_internos(df,reps,mid,wid)
    if mostrar_progreso:print('5/6 Diagnóstico A-Q y posibilidades de propagación.',flush=True)
    diag=diagnostico(df,reps,feats,cal,mg,wg,mid,wid,rev,pot,sha,vecinos)
    cand=feats.drop(columns=['i','j','Familia_A','Familia_B']).copy()
    # Evidencia legible para cada par; los Abstracts completos están en el plan.
    for letter,col in [('A','i'),('B','j')]:
        cand['Titulo_'+letter]=[reps[i]['Titulo'] for i in feats[col]]
        cand['Doi_'+letter]=[reps[i]['Doi'] for i in feats[col]]
        cand['row_ids_'+letter]=[' | '.join(map(str,reps[i]['row_ids'])) for i in feats[col]]
    return {'df':df,'sha':sha,'reps':reps,'feats':feats,'cal':cal,'thresholds':th,
       'plan':plan,'rev':rev,'pot':pot,'diag':diag,'cand':cand,'mg':mg,'wg':wg,'mid':mid,'wid':wid}


# ============================================================
# EJECUCIÓN DEL NOTEBOOK
# ============================================================
def localizar_raiz():
    if RAIZ_MANUAL is not None:
        root=Path(RAIZ_MANUAL).expanduser().resolve()
        if not (root/"04_Limpieza").is_dir():
            raise FileNotFoundError("RAIZ_MANUAL no contiene 04_Limpieza.")
        return root
    inicio=Path.cwd().resolve()
    for root in [inicio,*inicio.parents]:
        if (root/"04_Limpieza").is_dir() and (root/"notebooks").is_dir():return root
    raise FileNotFoundError("Abrir desde notebooks/ o configurar RAIZ_MANUAL con la raíz del repositorio.")

if modo not in {"DIAGNOSTICO","FUSION_FINAL"}:
    raise ValueError("modo debe ser DIAGNOSTICO o FUSION_FINAL.")
if not isinstance(vecinos_titulo,int) or vecinos_titulo<1:
    raise ValueError("vecinos_titulo debe ser un entero positivo.")
raiz=localizar_raiz()
archivo_entrada=raiz/"04_Limpieza"/"03_limpieza_bibliografica"/"autores_unam_limpios.csv"
carpeta_salida=raiz/"04_Limpieza"/"04_deduplicacion"
archivo_plan=carpeta_salida/"plan_manifestaciones.csv"
archivo_revision=carpeta_salida/"casos_revision_deduplicacion.csv"
archivo_salida=carpeta_salida/"autores_unam_deduplicados.csv"

# Solo se leen decisiones de ESTA estructura. No se importan las 182 decisiones
# antiguas ni se restaura la entrada desde una copia comprimida histórica.
plan_anterior=leer_csv(archivo_plan) if archivo_plan.exists() else None
revision_anterior=leer_csv(archivo_revision) if archivo_revision.exists() else None
if revision_anterior is not None and 'Caso_ID' not in revision_anterior:
    print("AVISO: el archivo de revisión existente es histórico; NO se usará como autoridad.")
    revision_anterior=None

resultado=analisis_completo(archivo_entrada,vecinos_titulo,plan_anterior,revision_anterior,mostrar_progreso=True)
entrada=resultado['df']
plan=resultado['plan']
candidatos=resultado['cand']
casos_revision=resultado['rev']
diagnostico_resumen=resultado['diag']
calibracion=resultado['cal']
potenciales=resultado['pot']

salidas={
    carpeta_salida/"diagnostico_deduplicacion.csv":diagnostico_resumen,
    carpeta_salida/"candidatos_deduplicacion.csv":candidatos,
    archivo_plan:plan,
    archivo_revision:casos_revision,
    carpeta_salida/"calibracion_deduplicacion.csv":calibracion,
    carpeta_salida/"potenciales_propagaciones.csv":potenciales,
}

if modo=="FUSION_FINAL":
    if not aprobar_diagnostico:
        raise ValueError("FUSIÓN BLOQUEADA: primero aprobar el diagnóstico. La primera entrega es no destructiva.")
    if sha256_diagnostico_aprobado!=resultado['sha']:
        raise ValueError("La huella aprobada está vacía o no corresponde a esta entrada.")
    salida,auditoria_fusion,auditoria_campos=fusion_final(
        entrada,resultado['reps'],resultado['feats'],plan,casos_revision,resultado['sha'])
    salidas[archivo_salida]=salida
    salidas[carpeta_salida/"auditoria_fusion.csv"]=auditoria_fusion
    salidas[carpeta_salida/"auditoria_campos_fusion.csv"]=auditoria_campos

# La entrada es inmutable en ambos modos. Todos los CSV se guardan/releen como texto.
print('6/6 Guardado protegido y relectura exacta de los CSV.',flush=True)
guardar_lote(salidas,actualizar=actualizar_archivos,protegidos=[archivo_entrada])
if hashlib.sha256(archivo_entrada.read_bytes()).hexdigest()!=resultado['sha']:
    raise RuntimeError("La entrada cambió durante la ejecución.")

print("=== FASE 06 -",modo,"===")
print("Entrada:",archivo_entrada.name)
print("SHA256:",resultado['sha'])
print("Filas originales:",len(entrada))
print("Títulos exactos:",entrada.Titulo.nunique())
print("Representaciones auxiliares:",len(plan))
print("Pares comparados (incluye controles):",len(candidatos))
print("Manifestaciones propuestas, NO definitivas:",len(resultado['mg']))
print("Casos de revisión:",len(casos_revision))
print("Pendientes:",int(casos_revision.Decision_manual.eq('').sum()))
print("Umbrales EXPLORATORIOS, no probabilidades:",resultado['thresholds'])
print("Ningún score sustituye las restricciones bibliográficas.")
print("Relectura exacta de todos los CSV: OK")
print("Archivo original intacto: OK")
if modo=="DIAGNOSTICO":
    print("NO se creó/actualizó autores_unam_deduplicados.csv, ni se reasignaron índices.")
    print("El archivo histórico deduplicado, si existe, NO es un resultado de esta ejecución.")
else:
    print("Base FINAL:",len(salida),"filas x",len(salida.columns),"columnas")
    print("Manifestaciones definitivas:",salida.indice.nunique())
print("Archivos generados/verificados:")
for p in salidas:print("-",p.relative_to(raiz))


1/6 Entrada validada: 5106 filas. Fuente_origen fuera de la lógica.
2/6 Comparando 9048 pares de perfiles, sin usar los índices históricos.
3/6 Pesos y umbrales exploratorios con evidencia interna.
4/6 Plan de identidad y conflictos por campo.
5/6 Diagnóstico A-Q y posibilidades de propagación.
6/6 Guardado protegido y relectura exacta de los CSV.
=== FASE 06 - DIAGNOSTICO ===
Entrada: autores_unam_limpios.csv
SHA256: e08804c671590b4f647ee57cbfde0a41a85995cc3ea3969885623274f81478e1
Filas originales: 5106
Títulos exactos: 585
Representaciones auxiliares: 970
Pares comparados (incluye controles): 9048
Manifestaciones propuestas, NO definitivas: 586
Casos de revisión: 594
Pendientes: 594
Umbrales EXPLORATORIOS, no probabilidades: {'obra': (-4.671897920271174, 9.797332307349247), 'manifestacion': (2.6496657967964095, 18.900105531810667)}
Ningún score sustituye las restricciones bibliográficas.
Relectura exacta de todos los CSV: OK
Archivo original intacto: OK
NO se creó/actualizó autores_u